# Soccer analytics on Colab — fine-tune, then run it on your own footage

Everything the GPU pod did, in one notebook. Train a 4-class detector, then process a clip end to end and download the annotated video.

**Runtime → Change runtime type → T4 GPU** before running anything.

Roughly 15 minutes total: ~9 min training, the rest download and inference.

---

### Why the fine-tune matters, precisely

A COCO checkpoint *does* have a ball class — `sports ball`, index 32. That surprises people, and an earlier version of these notes claimed otherwise. What COCO genuinely lacks is **`referee`**, and **`goalkeeper`** gets absorbed into `person`.

That second one is the expensive gap. Team assignment is k-means with k=2, and two clusters cannot represent four kits — outfield A, outfield B, two keepers, a referee. No hyperparameter fixes that. Detecting keepers and referees as their own classes removes them from the clustering input entirely, which makes k=2 the *right* model for what remains.

That is a **structural** fix rather than a tuning one, and the distinction is worth carrying: a parameter fix makes a wrong model less wrong; a structural fix makes the model right.

In [ ]:
# Pins carry hard-won reasons:
#   numpy<2.4        2.4 removed np.cross for 2-D vectors, which supervision uses
#   supervision<0.30 ByteTrack was removed in 0.30
!pip install -q "ultralytics>=8.4,<9" "supervision>=0.29,<0.30" "numpy<2.4" roboflow

import torch, ultralytics
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 1. The pipeline code

Cloned rather than pasted, so this notebook always runs the same code that is in the repo — including the goalkeeper handling, the majority-vote team assignment, NMS, and the frame-rate-aware tracker settings.

In [ ]:
import os, subprocess, sys

REPO = "/content/soccer-analytics"


def sh(cmd, cwd=None):
    """Plain subprocess rather than ! magic, so this cell is ordinary Python."""
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    return r.returncode


if os.path.isdir(REPO):
    sh("git pull --ff-only", cwd=REPO)
else:
    sh("git clone -q https://github.com/akshay131996/soccer-analytics.git", cwd="/content")

os.chdir(REPO)
sys.path.insert(0, "src")
sh("git log --oneline -1")

## 2. The dataset

[Roboflow football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc): `ball`, `goalkeeper`, `player`, `referee`.

You need a free Roboflow API key (Settings → API keys). Put it in **Colab Secrets** — the key icon in the left sidebar — named `ROBOFLOW_API_KEY`, and enable notebook access.

Do **not** paste it into a cell. A key typed into a notebook gets saved into the `.ipynb` and committed the moment you save. The fallback below prompts for it with masked input instead.

The version number is detected rather than hardcoded — it increments as the dataset is updated, and a stale number fails with an unhelpful error.

In [ ]:
import getpass

API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    pass
if not API_KEY:
    API_KEY = getpass.getpass("Roboflow API key (input hidden): ")

from roboflow import Roboflow

project = Roboflow(api_key=API_KEY).workspace("roboflow-jvuqo").project(
    "football-players-detection-3zvbc")

latest = sorted(int(str(v.version).split("/")[-1]) for v in project.versions())[-1]
print("using dataset version:", latest)

dataset = project.version(latest).download("yolov8")
DATA = os.path.join(dataset.location, "data.yaml")
print("data.yaml ->", DATA)

## 3. Verify the classes before spending a GPU hour

A class index is only meaningful relative to the model that produced it. Index 0 is `person` in COCO and `ball` here — the classes are alphabetical. Any code that hardcodes `class_id == 0` to mean "player" tracks the ball as a squad of players and still emits a full set of plausible statistics.

`src/pipeline.py` resolves classes **by name** and is immune. Assert it anyway: this check costs two seconds, and the same assumption invalidated an entire evaluation run in the sibling [traffic-lens](https://github.com/akshay131996/traffic-lens) project.

Note the split — `goalkeeper` is deliberately **not** in `PLAYER_NAMES`.

In [ ]:
import yaml
from pipeline import PLAYER_NAMES, GOALKEEPER_NAMES, BALL_NAMES, REFEREE_NAMES

cfg = yaml.safe_load(open(DATA))
names = cfg["names"]
names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else names

print("class index -> name")
for i, n in sorted(names.items()):
    print(f"  {i}: {n}")
print()

for label, wanted in (("outfield", PLAYER_NAMES), ("goalkeeper", GOALKEEPER_NAMES),
                      ("ball", BALL_NAMES), ("referee", REFEREE_NAMES)):
    got = [i for i, n in names.items() if str(n).lower() in wanted]
    print(f"  {label:11s} -> {got} {[names[i] for i in got]}")
    assert got, f"nothing matched {label}; the pipeline would not find it either"

print()
for split in ("train", "valid", "test"):
    d = os.path.join(dataset.location, split, "images")
    if os.path.isdir(d):
        print(f"  {split:6s}: {len(os.listdir(d))} images")
print("\nAll four resolve by name. Safe to train.")

## 4. Train

`imgsz=1280`, not the usual 640, for the reason this project keeps colliding with: **a football is ~10 px in a 1080p frame and effectively disappears at 640.** Training at the resolution you will infer at matters more here than in most fine-tunes, and it costs about 4× the compute of 640.

50 epochs with `patience=15`. A previous project in this series stopped at 20 epochs with the loss still falling — training ended because the cosine schedule had decayed the learning rate to zero, not because the model had converged. That was a budget artifact reported as a result. **Watch the curve, not the epoch count.**

Drop `batch` to 4 if you hit CUDA OOM on a T4.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATA,
    epochs=50,
    imgsz=1280,        # the ball is why
    batch=8,
    patience=15,
    project="runs",
    name="soccer_4class_1280",
    exist_ok=True,
    plots=True,
)
BEST = f"{results.save_dir}/weights/best.pt"
print("\nweights ->", BEST)

## 5. Per-class metrics — never read the mean alone

The *m* in mAP is "mean over classes", and it is where the most common misreading lives.

Reference numbers from a run of exactly this notebook (50 epochs, 298 train images):

| class | AP50 |
|---|---|
| player | **0.958** |
| referee | 0.853 |
| goalkeeper | 0.786 |
| **ball** | **0.499** |
| *mean* | *0.774* |

The headline 0.774 is carried almost entirely by `player`. **The ball scores half that** — and possession depends on it entirely. Expect the same shape; if your ball AP is near 0.5, treat any possession figure as indicative, not measured.

> One gotcha this notebook already handles: `split="val"`, not `"valid"`. The Roboflow export creates a directory called `valid/` but writes the key `val:` into `data.yaml`, and Ultralytics resolves the **key**. Passing `"valid"` crashes *after* training completes, which is a annoying way to lose a run.

In [ ]:
import json

m = YOLO(BEST)
metrics = m.val(data=DATA, imgsz=1280, split="val")   # "val", the yaml KEY

rows = []
print(f"\n{'class':<13}{'AP50':>9}{'AP50-95':>10}")
print("-" * 32)
for i, c in enumerate(metrics.box.ap_class_index):
    name = m.names[int(c)]
    ap50, ap = float(metrics.box.ap50[i]), float(metrics.box.ap[i])
    rows.append({"class": name, "AP50": round(ap50, 4), "AP50_95": round(ap, 4)})
    print(f"{name:<13}{ap50:>9.3f}{ap:>10.3f}")
print("-" * 32)
print(f"{'MEAN':<13}{metrics.box.map50:>9.3f}{metrics.box.map:>10.3f}")

os.makedirs("outputs", exist_ok=True)
json.dump({"per_class": rows, "mAP50": round(float(metrics.box.map50), 4),
           "mAP50_95": round(float(metrics.box.map), 4), "dataset_version": latest},
          open("outputs/finetune_results.json", "w"), indent=2)

worst = min(rows, key=lambda r: r["AP50"])
print(f"\nweakest class: {worst['class']} at AP50 {worst['AP50']} "
      f"({worst['AP50']/metrics.box.map50:.0%} of the mean)")

## 6. Your own footage

Two ways in. **Google Drive is the right choice for anything over ~100 MB** — `files.upload()` is unreliable at that size.

Set `SOURCE` to your clip, then run the next cell.

### The trim is not optional

Gameplay captures open on splash screens, replays and cutscenes. Team clustering is fitted on the **first** frames it can find players in — fit it on a title card and every downstream team label is garbage.

This bit me on the FC 26 capture: frame 300 was a "JUVENTUS vs FUT 26" screen with no pitch at all. `START_FRAME` skips past it. Scrub your clip and pick a frame where play is actually happening.

In [ ]:
# --- option A: Google Drive (recommended for large files) ---
# from google.colab import drive; drive.mount("/content/drive")
# SOURCE = "/content/drive/MyDrive/your_match.mp4"

# --- option B: direct upload (small clips only) ---
# from google.colab import files
# up = files.upload(); SOURCE = "/content/" + next(iter(up))

SOURCE = ""            # <-- set this
START_FRAME = 1200     # <-- skip the splash screen / cutscene
N_FRAMES = 1500        # ~25 s at 60 fps, ~60 s at 25 fps

assert SOURCE, "Set SOURCE to your video first (uncomment option A or B above)."

import cv2

cap = cv2.VideoCapture(SOURCE)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"source: {total} frames, {w}x{h} @ {fps:.2f} fps ({total/fps:.0f}s)")
assert START_FRAME < total, "START_FRAME is past the end of the clip"

SEG = "/content/segment.mp4"
cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)
out = cv2.VideoWriter(SEG, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
written = 0
while written < N_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    out.write(frame)
    written += 1
cap.release(); out.release()
print(f"segment: {written} frames -> {SEG}")

# Look at the first frame of the segment before spending inference on it.
from google.colab.patches import cv2_imshow
c = cv2.VideoCapture(SEG); ok, f0 = c.read(); c.release()
print("\nIf this is not live play, raise START_FRAME and re-run:")
cv2_imshow(cv2.resize(f0, (w // 2, h // 2)))

## 7. Run the pipeline

With the fine-tuned weights, `goalkeepers_excluded_from_clustering` should come back **true** — keepers and referees are now roles the detector reports, not guesses the clustering is forced to make. Four colours: two teams, gold keepers, green referees.

Leaving `keypoints=None` means no minimap, no distances, no possession. That is deliberate for a moving camera: the homography is fitted once and reused for the whole clip, so on panning footage a single calibration is silently wrong for most of it. Supply 5+ correspondences only if your camera is fixed — and with 5+, the homography will report its own reprojection error in metres.

In [ ]:
from pipeline import process_video

stats = process_video(
    source=SEG,
    output_path="outputs/annotated.mp4",
    weights=BEST,
    keypoints=None,
    imgsz=1280,
    conf=0.3,
    fit_frames=30,
    verbose=True,
)
print()
print(json.dumps(stats, indent=2))

## 8. Read the result honestly

The cell below flags the thing most likely to be wrong. On a 25 s clip with ~20 players on screen, a healthy `players_tracked` is somewhere near 20–40. Reference run: **182**.

That gap is real and mostly camera motion — ByteTrack matches on IoU and has no camera-motion compensation, so when the view pans, every box moves. Some of it is legitimate re-entry as players leave and rejoin frame.

**Without ground-truth tracks there is no way to separate those two**, which is exactly why the roadmap points at SoccerNet and HOTA rather than at more tuning. Track count is a proxy, and it is labelled as one.

In [ ]:
n = stats["players_tracked"]
roles = stats.get("roles_tracked", {})
print(f"tracks: {n}   mean length: {stats['mean_track_length_frames']} frames")
print(f"roles : {roles}")
print()
if not stats.get("goalkeepers_excluded_from_clustering"):
    print("! keepers were NOT excluded -- these weights have no goalkeeper class,")
    print("  so k=2 has forced them into an outfield team.")
if n > 60:
    print(f"! {n} tracks is well above the ~20 players present: fragmentation.")
    print("  Likely camera motion plus re-entry. This is a proxy, not a measurement.")
if not stats["has_pitch_mapping"]:
    print("! no keypoints -> no minimap, no distances, no possession (expected here).")

## 9. Download everything

The weights are the artifact worth keeping — a few MB, and they are what make every downstream feature real. Colab wipes the VM when the session ends.

In [ ]:
import shutil

os.makedirs("artifacts", exist_ok=True)
shutil.copy(BEST, "artifacts/yolo26n_soccer_4class.pt")
for f in ("results.png", "results.csv", "BoxPR_curve.png", "confusion_matrix_normalized.png"):
    p = os.path.join(results.save_dir, f)
    if os.path.exists(p):
        shutil.copy(p, f"artifacts/{f}")
for f in ("outputs/annotated.mp4", "outputs/finetune_results.json"):
    if os.path.exists(f):
        shutil.copy(f, f"artifacts/{os.path.basename(f)}")

print("\n".join(f"  {f}" for f in sorted(os.listdir("artifacts"))))

shutil.make_archive("soccer_artifacts", "zip", "artifacts")
sz = os.path.getsize("soccer_artifacts.zip") / 1e6
print(f"\nsoccer_artifacts.zip  ({sz:.0f} MB)")

try:
    from google.colab import files
    # The annotated video alone can be ~90 MB; the browser download can stall.
    # Grab the video separately and the rest as a zip.
    files.download("outputs/annotated.mp4")
    files.download("soccer_artifacts.zip")
except Exception as e:
    print(f"\nAuto-download unavailable ({e}). Use the Files pane on the left.")